# Transformers

### Contextualização

Antes de 2017, o processamento de linguagem natural (NLP) era dominado por Redes Neurais Recorrentes (RNNs) e LSTMs. Essas arquiteturas processam os dados sequencialmente (palavra por palavra). Isso trazia dois grandes problemas:

1.  **Impossibilidade de Paralelização:** Como o estado da palavra atual dependia da anterior, não era possível usar todo o poder das GPUs para processar uma frase inteira de uma vez.
2.  **Esquecimento de Longa Distância:** Em frases muito longas, a relação entre a primeira e a última palavra frequentemente se perdia, mesmo com mecanismos como LSTMs (problemática vista no laboratório de RNN e LSTM).

A arquitetura Transformer, introduzida no paper *"[Attention Is All You Need](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf?src_trk=em66c5166b7bad47.12959706424423622)"*, resolveu isso descartando a recorrência e baseando-se inteiramente no mecanismo de Atenção. Isso permite que o modelo olhe para a frase inteira simultaneamente.

Para desmistificar o funcionamento interno dessa arquitetura, uma das melhores referências visuais e o guia [The Illustrated Transformer by Jay Alammar](https://jalammar.github.io/illustrated-transformer/). O Autor comparar o modelo a uma pilha de blocos de processamento, onde os dados fluem de baixo para cima, ganhando representações cada vez mais ricas a cada camada. Recomenda-se a leitura do artigo para um melhor entendimento de como as arquiteturas Transformers funcionam.

É importante notar uma distinção crucial entre a aplicação completa e o que vai ser feito nesse laboratório. O Transformer original (proposto no artigo e exemplificado no material) possui duas partes, um codificador (que processa a entrada) e um decodificador (que gera a saída). O trabalho que será feito nesse laboratório para a tarefa de classificação de texto, será usado apenas a parte do codificador. Neste laboratório, será feito uma arquitetura de uma forma mais manual, e numa segunda parte, será utilizando uma arquitetura mais robusta chamada BERT, treinada usando *fine-tunning*.

Alguns pontos de implementação são úteis para destacar neste laboratório:

* **Positional Encoding**: Como foi removido a recorrência, o modelo não sabe nativamente a ordem das palavras. "O cão mordeu o homem" e "O homem mordeu o cão" pareceriam iguais para uma rede sem noção de sequência. Para resolver isso, será somado vetores matemáticos que marcam a posição de cada palavra na frase.
* **Self-attention**: Permite que cada palavra olhe para todas as outras palavras da frase simultaneamente para entender o contexto. Por exemplo, na frase "o banco estava fechado", o mecanismo de atenção ajuda o modelo a associar a palavra "banco" mais fortemente com "fechado" e "dinheiro" do que com "parque" ou "assento".

Em laboratórios anteriores, foi feita a construção de um modelo baseado em RNNs simples, onde foi possível observar na prática o problema do *Vanishing Gradient* (Desaparecimento do Gradiente), que impedia o aprendizado de sequências longas. A solução imediata foi a implementação de LSTMs (Long-Short-Term Memory), que mitiga esse problema através de portas de controle (*gates*) para reter informações importantes por mais tempo.

No entanto, embora as LSTMs sejam poderosas, elas ainda sofrem com a natureza sequencial do processamento, o que limita a capacidade do modelo de entender contextos globais complexos e torna o treinamento lento (em situações de aprendizagem mais complexas).

Agora, dando o próximo passo, será utilizado Transformers. Ao contrário das LSTMs, que leem o texto como uma fita continua, o Transformer consegue enxergar toda a sentença de uma vez. Isso não apenas acelera o treinamento através de paralelização, mas, crucialmente, permite capturar nuances semânticas e relações entre palavras distantes com uma precisão melhor da que foi feita com o modelo de LSTM em laboratórios passados.

### Conjunto de Dados (Recapitulação)

Para a validação deste modelo, reutilizaremos o conjunto de dados IMDB, cuja análise exploratória, balanceamento de classes e nuvens de palavras já foram detalhadamente discutidos no laboratório sobre RNN e LSTM.

Apenas como lembrete técnico para a configuração da arquitetura atual: trata-se de um problema de classificação binária (Positivo/Negativo). Para adequar os dados à entrada do Transformer e manter uma base de comparação justa com nossos modelos prévios (RNN/LSTM), foi mantido o vocabulário restrito às 20.000 palavras mais frequentes e foi padronizado o comprimento das sequências (*padding/truncating*) para 200 tokens. Isso garante que o foco do experimento seja a eficiência da arquitetura de atenção, isolando variáveis de pré-processamento.

In [ ]:
#!pip install tensorflow keras-preprocessing

In [9]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import torch.nn as nn
import torch

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb 

import numpy as np
import os

In [4]:
# Verificando compatibilidade com GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Recurso Gráfico: {device}")

Recurso Gráfico: cpu


In [5]:
# Configuração de hiperparâmetros
VOCAB_SIZE = 20000
MAXLEN = 200
EMBED_DIM = 32
NUM_HEADS = 2
FF_DIM = 32
BATCH_SIZE = 32
EPOCHS = 6
LEARNING_RATE = 1e-3

In [7]:
def prepare_data(vocab_size=20000, maxlen=200, batch_size=32):
    # prepara dataloaders de treino e validação do conjunto de dados IMDB

    # carrega o IMDB mantendo apenas as palavras mais frequentes do vocabulário
    (x_train, y_train), (x_val, y_val) = imdb.load_data(num_words=vocab_size)

    # padroniza todas as sequências para o mesmo comprimento
    x_train = pad_sequences(x_train, maxlen=maxlen)
    x_val = pad_sequences(x_val, maxlen=maxlen)
    
    # converte dados e rótulos para tensores PyTorch
    train_data = TensorDataset(torch.tensor(x_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.long))
    val_data = TensorDataset(torch.tensor(x_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.long))

    # cria dataloaders para iteração eficiente em batches
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    
    return train_loader, val_loader

In [10]:
# Preparação dos dados
train_loader, val_loader = prepare_data(VOCAB_SIZE, MAXLEN, BATCH_SIZE)

### Modelo Transformer

A arquitetura apresentada é um Transformer simplificado para classificação de sentimentos no dataset IMDB. Inicialmente, cada palavra da sequência de entrada é convertida em um vetor denso por meio de uma camada de *embedding* de *tokens*, que é somada a um embedding de posição para preservar a ordem das palavras. Em seguida, esses vetores são processados por um bloco Transformer, composto por um mecanismo de atenção que permite ao modelo relacionar cada palavra com todas as outras da sequência, seguido de uma rede feed-forward, ambas conectadas por conexões residuais e normalização para garantir estabilidade no treinamento. Após essa etapa, a informação contextual ao longo da sequência é resumida por uma operação de *pooling* global, gerando uma representação fixa do texto, que é então passada por camadas totalmente conectadas para produzir a predição final da classe de sentimento.

In [11]:
class TransformerBlock(nn.Module):
    # bloco codificador transformer com self-attention e feed-forward
    
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(TransformerBlock, self).__init__()
        
        # camada de atenção sobre a sequência de embeddings
        self.att = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        
        # rede feed forward aplicada a cada posição da sequência
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        
        # normalização e regularização para estabilidade
        self.layernorm1 = nn.LayerNorm(embed_dim)
        self.layernorm2 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        # camada de atenção
        attn_output, _ = self.att(x, x, x)  # Self-attention contextual
        attn_output = self.dropout1(attn_output)
        # Conexão residual
        out1 = self.layernorm1(x + attn_output) # Conexão residual
        
        # camada Feed Forward
        ffn_output = self.ffn(out1)  # Transformação não linear por token
        ffn_output = self.dropout2(ffn_output)
        
        return self.layernorm2(out1 + ffn_output) # Saída normalizada

In [12]:
class TokenAndPositionEmbedding(nn.Module):
    # soma embeddings de tokens e posições para entrada do transformer

    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim) # Embedding das palavras
        self.pos_emb = nn.Embedding(maxlen, embed_dim) # Embedding das posições
        self.maxlen = maxlen

    def forward(self, x):
        # x: (batch_size, seq_len)
        positions = torch.arange(0, self.maxlen, device=x.device) # Índices posicionais
        token_embeddings = self.token_emb(x) # (batch, seq_len, embed_dim)
        position_embeddings = self.pos_emb(positions) # (seq_len, embed_dim)
        
        return token_embeddings + position_embeddings # Soma token + posição


In [14]:
class IMDBTransformer(nn.Module):
    # transformer simplificado para classificação de sentimentos no dataset do IMDB

    def __init__(self, vocab_size, maxlen, embed_dim, num_heads, ff_dim, num_classes=2):
        super(IMDBTransformer, self).__init__()
        self.embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim) # Tokens + posições
        self.transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim) # Bloco Transformer

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1) # Pooling global na dimensão temporal
        self.dropout = nn.Dropout(0.1) # Regularização
        self.fc1 = nn.Linear(embed_dim, 20) # Projeção intermediária
        self.relu = nn.ReLU() # Não linearidade
        self.fc2 = nn.Linear(20, num_classes) # Camada de saída

    def forward(self, x):
        # x: (batch_size, seq_len)
        x = self.embedding_layer(x) # (batch, seq_len, embed_dim)
        x = self.transformer_block(x) # (batch, seq_len, embed_dim)

        x = x.permute(0, 2, 1) # (batch, embed_dim, seq_len)
        x = self.global_avg_pool(x) # (batch, embed_dim, 1)
        x = x.squeeze(-1) # (batch, embed_dim)

        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x) # (batch, num_classes)

        return x

In [15]:
# Inicializando o modelo
model = IMDBTransformer(VOCAB_SIZE, MAXLEN, EMBED_DIM, NUM_HEADS, FF_DIM).to(device)

In [16]:
# Função de custo e otimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

### Treinamento do Modelo

A célula abaixo tem a implementação manual do ciclo de treinamento do modelo construido. Diferente do que era feito com o Keras via TensorFlow, no PyTorch vamos fazer um controle explicito de cada etapa da atualização dos pesos. Esse processo é dividido em duas fases para cada época:

* **Fase de Treinamento**: Nessa fase, observe que é utilizado camadas de dropout (`self.dropout1`, `self.dropout2`) para evitar *overfitting*. O comando `model.train()` ativa essas camadas, fazendo com que neurônios aleatórios sejam desligados durante o treino.

Alguns outros detalhes merecem ser detacados:

a. `optimize.zero_grad()`: Limpa os gradientes antigos. No PyTorch, os gradientes se acumulam por padrão. Se não os zerarmos, a atualização dos pesos será errada.

b. `loss.backward()`: Calcula o gradiente da perda em relação a cada parâmetros (backpropagation).

c. `optmizer.step()`: Atualiza os pesos do modelo na direção oposta ao gradiente para minimizar o erro.

* **Fase de validação**: Nessa etapa, o modelo avalia dados nunca antes vistos. é chamado o `model.eval()`. Além disso, o dropout é desativado e ajusta a normalização. Para usar toda a capacidadade da rede e fazer uma melhor predição, é chamado `with torch.no_grad()`. Isso desliga os cálculos de gradientes, economizando memória VRAM e processamento.

In [17]:
# loop principal de treinamento por épocas
for epoch in range(EPOCHS):
    model.train()  # coloca o modelo em modo de treino (ativa dropout/batchnorm)
    total_loss = 0
    correct = 0
    total = 0
    
    # itera sobre os batches do conjunto de treino
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        
        inputs, targets = inputs.to(device), targets.to(device) # envia dados para CPU/GPU
        
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, targets) # calcula a função de perda
        
        optimizer.zero_grad() # zera os gradientes acumulados
        loss.backward() # backpropagation
        optimizer.step() # atualiza os pesos do modelo
        
        total_loss += loss.item() # acumula o valor da loss
        _, predicted = torch.max(outputs.data, 1) # obtém a classe prevista
        total += targets.size(0) # conta o número total de amostras
        correct += (predicted == targets).sum().item() # conta acertos no batch
        
        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")  

    train_acc = 100 * correct / total # calcula acurácia de treino
    print(f"Época [{epoch+1}/{EPOCHS}] Completa. Avg Loss: {total_loss/len(train_loader):.4f}, Acc Treino: {train_acc:.2f}%")

    # coloca o modelo em modo de avaliação
    model.eval()  
    with torch.no_grad(): # Desativa o cálculo de gradientes
        correct = 0
        total = 0
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device) # envia dados para CPU/GPU
            outputs = model(inputs) # forward pass na validação
            _, predicted = torch.max(outputs.data, 1) # obtém a classe prevista
            total += targets.size(0) # conta o número total de amostras
            correct += (predicted == targets).sum().item() # conta acertos na validação
    
        print(f"Acc Validação: {100 * correct / total:.2f}%\n") # exibe acurácia de validação


Epoch [1/6], Step [0/782], Loss: 0.6963
Epoch [1/6], Step [100/782], Loss: 0.6780
Epoch [1/6], Step [200/782], Loss: 0.6764
Epoch [1/6], Step [300/782], Loss: 0.5040
Epoch [1/6], Step [400/782], Loss: 0.3458
Epoch [1/6], Step [500/782], Loss: 0.4905
Epoch [1/6], Step [600/782], Loss: 0.4444
Epoch [1/6], Step [700/782], Loss: 0.4023
Época [1/6] Completa. Avg Loss: 0.5611, Acc Treino: 69.01%
Acc Validação: 79.14%

Epoch [2/6], Step [0/782], Loss: 0.3549
Epoch [2/6], Step [100/782], Loss: 0.3406
Epoch [2/6], Step [200/782], Loss: 0.2173
Epoch [2/6], Step [300/782], Loss: 0.3867
Epoch [2/6], Step [400/782], Loss: 0.3878
Epoch [2/6], Step [500/782], Loss: 0.3162
Epoch [2/6], Step [600/782], Loss: 0.2420
Epoch [2/6], Step [700/782], Loss: 0.4631
Época [2/6] Completa. Avg Loss: 0.3565, Acc Treino: 84.62%
Acc Validação: 83.89%

Epoch [3/6], Step [0/782], Loss: 0.1687
Epoch [3/6], Step [100/782], Loss: 0.3676
Epoch [3/6], Step [200/782], Loss: 0.1534
Epoch [3/6], Step [300/782], Loss: 0.3963
Ep

### Avaliação do Modelo

Nesta etapa, o modelo treinado é avaliado utilizando textos que não fizeram parte do processo de treinamento, com o objetivo de verificar sua capacidade de generalização. Cada texto passa por um pré-processamento que inclui tokenização, conversão das palavras em índices numéricos com base no vocabulário do IMDB e aplicação de padding para garantir um tamanho fixo de entrada. Em seguida, o modelo realiza a inferência em modo de avaliação, produzindo probabilidades para cada classe de sentimento. A classe com maior probabilidade é selecionada como predição final, juntamente com um valor de confiança, permitindo analisar qualitativamente o desempenho do modelo em novos exemplos.

In [19]:
def get_imdb_word_index():
    # carrega e ajusta o índice de palavras do imdb para compatibilidade com o treino

     # carrega o dicionário palavra - índice original do IMDB
    word_index = imdb.get_word_index() 
    
    # desloca os índices para reservar tokens especiais
    word_index = {k: (v + 3) for k, v in word_index.items()}
    
    word_index["<PAD>"] = 0 # token usado para padding das sequências
    word_index["<START>"] = 1 # token que indica início da sequência
    word_index["<UNK>"] = 2 # token para palavras desconhecidas
    word_index["<UNUSED>"] = 3 # token reservado e não utilizado
    
    return word_index

In [20]:
def predict_sentiment(model, texts, word_index, maxlen=200):
    # realiza a predição de sentimento para uma lista de textos

    # coloca o modelo em modo de avaliação (desativa dropout)
    model.eval()  
    sequences = [] # lista para armazenar as sequências tokenizadas
    
    for text in texts:
        # tokeniza o texto e remove pontuações básicas
        words = text.lower().replace('.', '').replace(',', '').replace('!', '').replace('?', '').split()
        
        # converte cada palavra em seu índice correspondente no vocabulário
        sequence = [word_index.get(w, 2) for w in words]
        
        # aplica padding ou truncamento para garantir tamanho fixo
        if len(sequence) < maxlen:
            sequence = [0] * (maxlen - len(sequence)) + sequence
        else:
            sequence = sequence[-maxlen:]
            
        sequences.append(sequence) # adiciona a sequência processada à lista
    
    # converte as sequências para tensor e envia para CPU/GPU
    input_tensor = torch.tensor(sequences, dtype=torch.long).to(device)
    
    # executa a inferência sem calcular gradientes
    with torch.no_grad():
        logits = model(input_tensor) # obtém as saídas brutas do modelo
        probabilities = torch.softmax(logits, dim=1) # converte logits em probabilidades
        predictions = torch.argmax(probabilities, dim=1) # seleciona a classe mais provável
        
    results = [] # lista para armazenar os resultados finais
    class_names = ["Negative", "Positive"] # rótulos das classes
    
    for i, text in enumerate(texts):
        pred_idx = predictions[i].item() # índice da classe prevista
        confidence = probabilities[i][pred_idx].item() # confiança da predição

        results.append({
            "text": text,
            "sentiment": class_names[pred_idx],
            "negative_probability": f"{probabilities[i][0].item()*100:.2f}%",
            "positive_probability": f"{probabilities[i][1].item()*100:.2f}%",
            "confidence": f"{confidence*100:.2f}%"
        })
        
    return results

In [21]:
word_index = get_imdb_word_index()

In [23]:
my_reviews = [
    "This movie was absolutely fantastic, the acting was great and the plot was amazing.",
    "I hated this film. It was boring, slow, and a complete waste of time.",
    "The visual effects were incredible and the sound design was top tier, but the story was empty and boring.",
    "I really wanted to love this movie. The trailer looked like a masterpiece, but the actual film was a huge letdown.",
    "If you are a fan of mindless action with zero plot, this is perfect. Otherwise, stay away.",
    "Oh, brilliant. Another superhero movie with the exact same plot as the last fifty ones. Just what we needed.",
    "It was not the best movie I have ever seen, but it was certainly not the worst. It was just okay.",
    "I cannot say I didn't enjoy it. It was actually better than I expected.",
    "At first I thought this movie was going to be terrible, but the ending was absolutely perfect.",
    "Funny at times, yet I wouldn't watch it again.",
    "Nice visuals and great performances, but the plot was completely confusing, waste of time and terrible."
]

predictions = predict_sentiment(model, my_reviews, word_index, MAXLEN)

for p in predictions:
    print(f"Análise: '{p['text']}'")
    print(f"Predição: {p['sentiment']} (Probabilidade Positiva: {p['positive_probability']} | Probabilidade Negativa: {p['negative_probability']})")
    print("-" * 50)

Análise: 'This movie was absolutely fantastic, the acting was great and the plot was amazing.'
Predição: Positive (Probabilidade Positiva: 99.07% | Probabilidade Negativa: 0.93%)
--------------------------------------------------
Análise: 'I hated this film. It was boring, slow, and a complete waste of time.'
Predição: Negative (Probabilidade Positiva: 0.07% | Probabilidade Negativa: 99.93%)
--------------------------------------------------
Análise: 'The visual effects were incredible and the sound design was top tier, but the story was empty and boring.'
Predição: Negative (Probabilidade Positiva: 24.13% | Probabilidade Negativa: 75.87%)
--------------------------------------------------
Análise: 'I really wanted to love this movie. The trailer looked like a masterpiece, but the actual film was a huge letdown.'
Predição: Negative (Probabilidade Positiva: 15.18% | Probabilidade Negativa: 84.82%)
--------------------------------------------------
Análise: 'If you are a fan of mindless 